# Bleeding SKU & Net Margin Prediction (Random Forest)
This model classifies whether a product will bleed profit margin and identifies feature importance (e.g., PPC vs FBA impacts).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import classification_report, mean_squared_error
import matplotlib.pyplot as plt

# Load prepared data
df = pd.read_csv('ml_ready_data.csv')

## Prepare Features (X) and Target (y)

In [ ]:
# Predict next day's profitability based on today's metrics
df = df.sort_values(['Product_Key', 'Date'])
df['Next_Day_Margin'] = df.groupby('Product_Key')['Net Margin'].shift(-1)
df['Next_Day_Bleeding'] = (df['Next_Day_Margin'] < 0.10).astype(int)

df_clean = df.dropna(subset=['Next_Day_Margin'])

features = ['Sessions', 'Page Views', 'Clicks', 'PPC Cost', 'FBA Fees', 'Promo Amount', 'Unit Session %', 'Revenue_7d_avg']
X = df_clean[features].fillna(0)
y_reg = df_clean['Next_Day_Margin']
y_clf = df_clean['Next_Day_Bleeding']

X_train, X_test, y_train, y_test = train_test_split(X, y_clf, test_size=0.2, random_state=42)

## Train Random Forest Classifier

In [ ]:
clf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Bleeding SKU Detection Report:")
print(classification_report(y_test, y_pred))

## Feature Importance

In [ ]:
importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Drivers of Margin Bleeding")
plt.bar(range(X.shape[1]), importances[indices], align="center")
plt.xticks(range(X.shape[1]), [features[i] for i in indices], rotation=45)
plt.tight_layout()
plt.show()

## Export Predictions for Dashboard

In [ ]:
# Predict on latest available date
latest_data = df.groupby('Product_Key').last().reset_index()
X_latest = latest_data[features].fillna(0)
latest_data['Predicted_Bleeding_Risk'] = clf.predict_proba(X_latest)[:, 1]

risk_dashboard = latest_data[['Product_Key', 'Title', 'Predicted_Bleeding_Risk']]
risk_dashboard.to_csv('sku_risk_dashboard.csv', index=False)
print("Exported 'sku_risk_dashboard.csv'")